# PCOS Prediction — 03: Preprocessing & Feature Engineering

This is notebook **3 of 4** in the PCOS prediction pipeline. It picks up from `data/pcos_cleaned.csv` and the statistical findings established in `02_Exploratory_Data_Analysis.ipynb`.

## This notebook's structure
| Section | What it does |
|---|---|
| 11 | Feature selection — gap-testing, candidate set, systematic multicollinearity check, engineering, log transform, encoding |
| 12 | Train/test split |
| 13 | Scaling |
| 14 | Handling class imbalance (strategy note — implemented per-model in notebook 4) |

**Output:** `data/processed/` — `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, `X_train_scaled.csv`, `X_test_scaled.csv`. These are the **full 19-feature, unreduced** matrices; Lasso-based reduction for Logistic Regression specifically happens in `04_Model_Building_and_Evaluation.ipynb`, since it's a modeling decision, not a preprocessing one, and Random Forest / XGBoost use the full feature set.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)

df_clean = pd.read_csv("../data/pcos_cleaned.csv")
target_col = 'PCOS (Y/N)'

print(f"Loaded data/pcos_cleaned.csv — shape: {df_clean.shape}")

Loaded data/pcos_cleaned.csv — shape: (541, 42)


## Section 11 — Preparing Data for Modeling

Before fitting anything, we make firm, defensible decisions about missing data, redundant features, and encoding. Each decision below traces back to a specific finding earlier in the notebook — nothing here is assumed for the first time.

### 11.0 — Mann-Whitney U test for additional numeric features

Section 8.2 tested the main clinically obvious numeric variables. This section applies the same Mann-Whitney U procedure to the remaining numeric measurements that were not included earlier — such as follicle size, waist/hip measurements, vitamin and hormone panels, and vitals.

This is not a different statistical test. It is the same test applied to the remaining numeric features so that nothing is missed accidentally.

In [2]:
# Mann-Whitney U test for the remaining numeric features
# Same test as Section 8.2, just applied to the features not included there.

untested_numeric = [
    'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)',
    'Waist(inch)', 'Hip(inch)',
    'Vit D3 (ng/mL)', 'PRG(ng/mL)',
    'I beta-HCG(mIU/mL)', 'II beta-HCG(mIU/mL)',
    'Pulse rate(bpm)', 'RR (breaths/min)',
    'No. of aborptions', 'Marraige Status (Yrs)',
    'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)'
]

g0 = df_clean[df_clean[target_col] == 0]
g1 = df_clean[df_clean[target_col] == 1]

gap_results = []
for col in untested_numeric:
    a = g0[col].dropna()
    b = g1[col].dropna()
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    gap_results.append({
        'Feature': col,
        'Median (No PCOS)': round(a.median(), 3),
        'Median (PCOS)': round(b.median(), 3),
        'p-value': p,
        'Significant (p<0.05)': p < 0.05
    })

gap_df = pd.DataFrame(gap_results).sort_values('p-value')
gap_df_display = gap_df.copy()
gap_df_display['p-value'] = gap_df_display['p-value'].apply(lambda x: f'{x:.2e}')
print(gap_df_display.to_string(index=False))

              Feature  Median (No PCOS)  Median (PCOS)  p-value  Significant (p<0.05)
          Waist(inch)            34.000          35.00 4.83e-05                  True
            Hip(inch)            38.000          39.00 1.59e-04                  True
Marraige Status (Yrs)             7.000           6.00 3.39e-03                  True
      Pulse rate(bpm)            72.000          72.00 5.80e-03                  True
 Avg. F size (L) (mm)            15.000          16.00 9.31e-03                  True
 Avg. F size (R) (mm)            15.000          16.00 2.64e-02                  True
   I beta-HCG(mIU/mL)            13.735          70.53 6.85e-02                 False
       Vit D3 (ng/mL)            26.300          25.45 2.30e-01                 False
     RR (breaths/min)            18.000          20.00 2.81e-01                 False
           PRG(ng/mL)             0.310           0.32 3.86e-01                 False
    No. of aborptions             0.000           0.00

**Result:** several additional numeric features are significant, including Waist(inch), Hip(inch), Marraige Status (Yrs), Pulse rate(bpm), and both Avg. F size variables.

These are now on the same footing as the features identified in Section 8.2. The p-values from Section 11.0 matter just as much as the p-values from Section 8.2 because the same Mann-Whitney U test is being used.

At this stage, we are only identifying candidate features. Redundancy and final selection are handled later.

## 11.1 — Building the Candidate Feature Set

The previous statistical analyses identified features that show a significant association with PCOS.

At this stage, all statistically significant features are collected into a **candidate feature set**.

Important:

- Features are included **solely based on statistical significance**.
- **No feature is removed yet because of redundancy or multicollinearity.**
- Redundancy analysis is performed separately in the next section before model development.

This separation keeps the statistical screening stage independent from the feature selection stage.

In [3]:
# ============================================================
# Section 11.1 - Candidate features after statistical screening
# ============================================================

# Numeric features that were statistically significant
# in the Mann-Whitney U tests (Sections 8.2 and 11.0)

candidate_numeric = [
    'Follicle No. (L)',
    'Follicle No. (R)',
    'AMH(ng/mL)',
    'BMI',
    'Weight (Kg)',
    'Age (yrs)',
    'Cycle length(days)',
    'Endometrium (mm)',
    'FSH/LH',
    'FSH(mIU/mL)',
    'Hb(g/dl)',
    'Waist(inch)',
    'Hip(inch)',
    'Marraige Status (Yrs)',
    'Pulse rate(bpm)',
    'Avg. F size (L) (mm)',
    'Avg. F size (R) (mm)'
]

# Categorical features significant by Chi-square test

candidate_categorical = [
    'Cycle(R/I)',
    'Skin darkening (Y/N)',
    'hair growth(Y/N)',
    'Weight gain(Y/N)',
    'Fast food (Y/N)',
    'Pimples(Y/N)',
    'Hair loss(Y/N)'
]

print(f"Candidate numeric features: {len(candidate_numeric)}")
print(f"Candidate categorical features: {len(candidate_categorical)}")

Candidate numeric features: 17
Candidate categorical features: 7


## 11.2 — Redundancy Analysis Before Modeling

Statistical significance alone is **not sufficient** for selecting model features.

Some variables may contain overlapping information because they measure similar biological quantities or one variable is mathematically derived from another.

Therefore, before model development, we examine correlations among the candidate features to identify potential redundancy.

Although **LH** was not retained as a candidate feature after statistical screening, it is temporarily included here because the **FSH/LH ratio is mathematically derived from both FSH and LH**. This allows the redundancy analysis to be interpreted correctly without affecting the candidate feature set.

In [4]:
# ============================================================
# Section 11.2 - Redundancy / Multicollinearity Analysis
# ============================================================

# LH is included ONLY for redundancy analysis.
# Although LH was not selected as a candidate feature,
# it is biologically linked with FSH and the derived FSH/LH ratio.

# ============================================================
# Candidate features + LH (LH is included ONLY for redundancy analysis)
# ============================================================

redundancy_numeric = candidate_numeric + ['LH(mIU/mL)']

redundancy_all_cols = redundancy_numeric + candidate_categorical

candidate_all_cols = redundancy_all_cols

missing_in_candidates = df_clean[candidate_all_cols].isnull().sum()
missing_in_candidates = missing_in_candidates[missing_in_candidates > 0]

print("Columns with missing values among candidate features:")
print(missing_in_candidates)

df_model = df_clean.dropna(subset=missing_in_candidates.index.tolist()).copy()

print(f"\nRows before dropping: {len(df_clean)}")
print(f"Rows after dropping:  {len(df_model)}")
print(f"Rows lost: {len(df_clean)-len(df_model)} "
      f"({(len(df_clean)-len(df_model))/len(df_clean)*100:.2f}%)")

print("\nClass balance before vs after (should barely move):")
print("Before:", df_clean[target_col].value_counts(normalize=True).round(3).to_dict())
print("After: ", df_model[target_col].value_counts(normalize=True).round(3).to_dict())

Columns with missing values among candidate features:
AMH(ng/mL)               1
Marraige Status (Yrs)    1
Pulse rate(bpm)          2
Cycle(R/I)               1
Fast food (Y/N)          1
dtype: int64

Rows before dropping: 541
Rows after dropping:  535
Rows lost: 6 (1.11%)

Class balance before vs after (should barely move):
Before: {0: 0.673, 1: 0.327}
After:  {0: 0.673, 1: 0.327}


### 11.3 — Systematic multicollinearity check

Rather than eyeballing the Section 9 heatmap for pairs that look related, we compute the Spearman correlation matrix for *every* candidate numeric feature (including `LH`, to justify dropping it rather than assume it) and flag anything with `|ρ| > 0.7` — a standard rule-of-thumb threshold for "these two features are carrying largely the same information." Pairs between 0.55 and 0.7 are noted for awareness but aren't automatically actioned.

In [5]:
# ============================================================
# Section 11.3 - Systematic multicollinearity check
# ============================================================

# Include LH ONLY for redundancy analysis.
# LH is not a final candidate feature, but is examined here
# because the FSH/LH ratio is mathematically derived from FSH and LH.

redundancy_numeric = candidate_numeric + ['LH(mIU/mL)']

collin_corr = df_model[redundancy_numeric].corr(method='spearman')

cols = collin_corr.columns

flagged = []
noted = []

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        rho = collin_corr.iloc[i, j]

        if abs(rho) > 0.70:
            flagged.append((cols[i], cols[j], round(rho, 3)))

        elif abs(rho) > 0.55:
            noted.append((cols[i], cols[j], round(rho, 3)))

flagged.sort(key=lambda x: -abs(x[2]))
noted.sort(key=lambda x: -abs(x[2]))

print("Flagged for action (|rho| > 0.70):\n")

if len(flagged) == 0:
    print("None")
else:
    for f in flagged:
        print(f"{f[0]:30s} <-> {f[1]:25s} rho = {f[2]}")

print("\nNoted for awareness (0.55 < |rho| <= 0.70):\n")

if len(noted) == 0:
    print("None")
else:
    for f in noted:
        print(f"{f[0]:30s} <-> {f[1]:25s} rho = {f[2]}")

Flagged for action (|rho| > 0.70):

BMI                            <-> Weight (Kg)               rho = 0.879
FSH/LH                         <-> LH(mIU/mL)                rho = -0.827
Follicle No. (L)               <-> Follicle No. (R)          rho = 0.823
Waist(inch)                    <-> Hip(inch)                 rho = 0.817

Noted for awareness (0.55 < |rho| <= 0.70):

Age (yrs)                      <-> Marraige Status (Yrs)     rho = 0.635
Weight (Kg)                    <-> Waist(inch)               rho = 0.628
Weight (Kg)                    <-> Hip(inch)                 rho = 0.628
Avg. F size (L) (mm)           <-> Avg. F size (R) (mm)      rho = 0.605
BMI                            <-> Waist(inch)               rho = 0.59
BMI                            <-> Hip(inch)                 rho = 0.578


### 11.4 — Resolving the flagged pairs

| Flagged pair | ρ | Resolution |
|---|---|---|
| `BMI` vs `Weight (Kg)` | 0.879 | Keep `BMI` (p=5.16e-06 vs Weight's p=5.84e-06 in Section 8.2 — marginally stronger, and BMI is the height-normalized clinical standard). Drop `Weight`. |
| `LH(mIU/mL)` vs `FSH/LH` | -0.827 | `LH` was already not significant on its own (Section 8.2, p=0.354). Drop `LH`. |
| `Follicle No. (L)` vs `(R)` | 0.823 | Engineer `Follicle_Avg` (mean of both ovaries) — keeps the signal from both without the collinearity. Drop both raw counts. |
| `Waist(inch)` vs `Hip(inch)` | 0.817 | Keep `Waist` (p=4.83e-05 vs Hip's p=1.59e-04 — stronger). Drop `Hip`. |

**A judgment call beyond the correlation check:** `FSH(mIU/mL)` is only weakly correlated with `FSH/LH` (ρ ≈ -0.08, from Section 9) so the systematic check doesn't force a drop — but we drop it anyway. `FSH/LH` is already the clinically standard composite marker for this hormone panel (an elevated LH:FSH ratio is the classic PCOS sign), and keeping raw `FSH` alongside it would represent the same lab measurement twice under two different names.

**Below the 0.70 action threshold, kept as-is:**
- `Avg. F size (L)` vs `(R)` (ρ = 0.605) — below the hard threshold, but we average these into `Avg_F_size` anyway, for the same reason as the follicle counts: they're a bilateral anatomical measurement pair, and treating both ovary-pair features the same way is more consistent than applying one rule to follicle count and a different rule to follicle size.
- `Age (yrs)` vs `Marraige Status (Yrs)` (ρ = 0.635) — expected (longer-married patients tend to be older) but not strong enough to act on; both are kept.
- `BMI` vs `Waist`/`Hip` (ρ ≈ 0.58-0.59) — moot for Hip (already dropped above); BMI and Waist both stay, since 0.59 is well under the threshold and they capture related but distinct information (overall adiposity vs central/abdominal fat distribution).

In [6]:
# Engineer bilateral-measurement averages
df_model['Follicle_Avg'] = df_model[['Follicle No. (L)', 'Follicle No. (R)']].mean(axis=1)
df_model['Avg_F_size'] = df_model[['Avg. F size (L) (mm)', 'Avg. F size (R) (mm)']].mean(axis=1)

drop_cols = [
    'Follicle No. (L)', 'Follicle No. (R)',        # replaced by Follicle_Avg
    'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', # replaced by Avg_F_size
    'Weight (Kg)',                                   # redundant with BMI (rho=0.878)
    'Hip(inch)',                                     # redundant with Waist (rho=0.818)
    'FSH(mIU/mL)', 'LH(mIU/mL)',                     # superseded by FSH/LH
]
df_model = df_model.drop(columns=drop_cols)

print("Dropped:", drop_cols)
print("New shape:", df_model.shape)
print("\nEngineered features (first 5 rows):")
print(df_model[['Follicle_Avg', 'Avg_F_size']].head())

Dropped: ['Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Weight (Kg)', 'Hip(inch)', 'FSH(mIU/mL)', 'LH(mIU/mL)']
New shape: (535, 36)

Engineered features (first 5 rows):
   Follicle_Avg  Avg_F_size
0           3.0        18.0
1           4.0        14.5
2          14.0        19.0
3           2.0        14.5
4           3.5        15.0


### 11.5 — Final candidate feature list

| Kept | Why |
|---|---|
| `Follicle_Avg` | strongest signal in the dataset; engineered in 11.4 |
| `Avg_F_size` | significant in 11.0 (gap test); engineered in 11.4 |
| `AMH(ng/mL)` | Mann-Whitney p = 3.9e-08 |
| `BMI` | Mann-Whitney p = 5.2e-06; kept over `Weight` (11.4) |
| `Age (yrs)` | Mann-Whitney p = 1.7e-05 |
| `Cycle length(days)` | Mann-Whitney p = 6.2e-09 |
| `Endometrium (mm)` | Mann-Whitney p = 4.5e-03 |
| `FSH/LH` | Mann-Whitney p = 6.0e-03 (survives despite the t-test missing it, Section 8.2) |
| `Hb(g/dl)` | Mann-Whitney p = 0.025 |
| `Waist(inch)` | gap-test p = 4.8e-05; kept over `Hip` (11.4) |
| `Marraige Status (Yrs)` | gap-test p = 3.4e-03; correlated with Age (ρ=0.64) but below action threshold |
| `Pulse rate(bpm)` | gap-test p = 4.3e-03 (after the Section 3.6b data-quality fix) |
| `Cycle(R/I)`, symptom flags (6) | Chi-square p < 0.0001, Cramér's V 0.17-0.48 |

| Dropped | Why |
|---|---|
| `LH`, `TSH`, `PRL`, `Waist:Hip Ratio`, `Height`, `RBS` | not significant (Section 8.2) |
| `Blood Group`, `Reg.Exercise`, `Pregnant` | not significant (Section 8.3) |
| `Vit D3`, `PRG`, `I/II beta-HCG`, `RR`, `No. of aborptions`, `BP Systolic/Diastolic` | not significant (11.0 gap test) |
| `Weight`, `Hip`, `FSH`, raw `Follicle L/R`, raw `Avg. F size L/R` | redundant with a kept feature — resolved in 11.4 |


In [7]:
significant_numeric = [
    'Follicle_Avg', 'Avg_F_size', 'AMH(ng/mL)', 'BMI', 'Age (yrs)',
    'Cycle length(days)', 'Endometrium (mm)', 'FSH/LH', 'Hb(g/dl)',
    'Waist(inch)', 'Marraige Status (Yrs)', 'Pulse rate(bpm)'
]

significant_categorical = [
    'Cycle(R/I)', 'Skin darkening (Y/N)', 'hair growth(Y/N)',
    'Weight gain(Y/N)', 'Fast food (Y/N)', 'Pimples(Y/N)', 'Hair loss(Y/N)'
]

feature_cols = significant_numeric + significant_categorical
X = df_model[feature_cols].copy()
y = df_model[target_col].copy()

print(f"Final feature count: {len(feature_cols)}")
print(feature_cols)
print(f"\nX shape: {X.shape}, y shape: {y.shape}")

Final feature count: 19
['Follicle_Avg', 'Avg_F_size', 'AMH(ng/mL)', 'BMI', 'Age (yrs)', 'Cycle length(days)', 'Endometrium (mm)', 'FSH/LH', 'Hb(g/dl)', 'Waist(inch)', 'Marraige Status (Yrs)', 'Pulse rate(bpm)', 'Cycle(R/I)', 'Skin darkening (Y/N)', 'hair growth(Y/N)', 'Weight gain(Y/N)', 'Fast food (Y/N)', 'Pimples(Y/N)', 'Hair loss(Y/N)']

X shape: (535, 19), y shape: (535,)


### 11.6 — Log-transforming remaining skewed features

Rather than hardcoding which features get log-transformed, we check the skewness of every feature actually in `X` and apply `log1p` to any with `|skewness| > 1` — the same threshold used in Section 6.3. This also makes sure we don't miss a skewed feature that entered the model via the 11.0 gap test rather than Section 8.2. Logistic Regression assumes a roughly linear relationship between each feature and the log-odds of the target, so compressing a long right tail helps it learn that relationship; Random Forest doesn't strictly need this (it splits on rank order, not raw magnitude), but the transform doesn't hurt it either, so we apply it once for both models.

In [8]:
skew_before = X[significant_numeric].skew()
to_transform = skew_before[skew_before.abs() > 1].index.tolist()

print("Skewness of numeric features before transform:")
print(skew_before.round(2).sort_values(ascending=False))
print(f"\nFeatures with |skew| > 1 -> log1p transform: {to_transform}")

for col in to_transform:
    before = X[col].skew()
    X[col] = np.log1p(X[col])
    after = X[col].skew()
    print(f"{col}: skew {before:.2f} -> {after:.2f}")

Skewness of numeric features before transform:
FSH/LH                   21.35
AMH(ng/mL)                3.29
Pulse rate(bpm)           1.23
Marraige Status (Yrs)     1.14
Cycle length(days)        0.83
Follicle_Avg              0.79
Hb(g/dl)                  0.74
Age (yrs)                 0.36
Endometrium (mm)          0.26
BMI                       0.26
Waist(inch)               0.19
Avg_F_size               -0.67
dtype: float64

Features with |skew| > 1 -> log1p transform: ['AMH(ng/mL)', 'FSH/LH', 'Marraige Status (Yrs)', 'Pulse rate(bpm)']
AMH(ng/mL): skew 3.29 -> 0.36
FSH/LH: skew 21.35 -> 2.45
Marraige Status (Yrs): skew 1.14 -> -0.24
Pulse rate(bpm): skew 1.23 -> 1.18


### 11.7 — Encoding `Cycle(R/I)`

`Cycle(R/I)` currently holds the raw codes 2 (Regular) and 4 (Irregular). These aren't ordinal — Irregular isn't "more" than Regular — but since there are only two categories, recoding to 0/1 is mathematically equivalent to one-hot encoding here and simpler to read. `Blood Group` needs no encoding decision since it was already dropped in Section 11.5 for having no relationship with the target.

In [9]:
X['Cycle(R/I)'] = X['Cycle(R/I)'].map({2: 0, 4: 1})  # 0 = Regular, 1 = Irregular
print(X['Cycle(R/I)'].value_counts(dropna=False))
assert X['Cycle(R/I)'].isna().sum() == 0, "Unmapped category found -- check for stray codes"

Cycle(R/I)
0    387
1    148
Name: count, dtype: int64


## Section 12 — Train/Test Split

We split **before** scaling and before doing anything else that "looks at" the full dataset, to avoid leaking test-set information into training. `stratify=y` keeps the ~67/33 class ratio consistent in both splits — a plain random split could otherwise hand the test set a different imbalance than what the model was trained on, making the evaluation misleading.

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print("\nClass balance — train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Class balance — test: ", y_test.value_counts(normalize=True).round(3).to_dict())

Train shape: (428, 19), Test shape: (107, 19)

Class balance — train: {0: 0.673, 1: 0.327}
Class balance — test:  {0: 0.673, 1: 0.327}


## Section 13 — Feature Scaling

`StandardScaler` is **fit only on the training data**, then applied to both train and test. Fitting it on the full dataset (including test) would leak the test set's mean and variance into training — a subtle form of data leakage that makes test performance look better than it would be on genuinely unseen data. Random Forest doesn't need scaling to work correctly, but scaling it too doesn't hurt performance, and it keeps both models on an identical feature set for a fair comparison.

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print("Train mean after scaling (should be ~0 for every column):")
print(X_train_scaled.mean().round(2))

Train mean after scaling (should be ~0 for every column):
Follicle_Avg            -0.0
Avg_F_size               0.0
AMH(ng/mL)               0.0
BMI                     -0.0
Age (yrs)               -0.0
Cycle length(days)      -0.0
Endometrium (mm)         0.0
FSH/LH                   0.0
Hb(g/dl)                 0.0
Waist(inch)              0.0
Marraige Status (Yrs)    0.0
Pulse rate(bpm)         -0.0
Cycle(R/I)               0.0
Skin darkening (Y/N)     0.0
hair growth(Y/N)         0.0
Weight gain(Y/N)         0.0
Fast food (Y/N)          0.0
Pimples(Y/N)            -0.0
Hair loss(Y/N)           0.0
dtype: float64


## Section 14 — Handling Class Imbalance

Section 6.1 established the target is ~67/33 imbalanced. A model that always predicts "No PCOS" scores 67% accuracy while learning nothing, so two things need to change:

1. **`class_weight='balanced'`** in both models. This up-weights errors on the minority class (PCOS) during training, proportional to how rare it is, so the model isn't implicitly rewarded for ignoring it. This is simpler than resampling techniques like SMOTE and is a reasonable baseline choice for a dataset this size.
2. **Never evaluate with accuracy alone.** Precision, recall, F1, and ROC-AUC are used throughout Sections 15–17 because they score each class's performance separately rather than averaging over an imbalanced pool.

In [12]:
import os
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=True)
X_test.to_csv("../data/processed/X_test.csv", index=True)
y_train.to_csv("../data/processed/y_train.csv", index=True)
y_test.to_csv("../data/processed/y_test.csv", index=True)
X_train_scaled.to_csv("../data/processed/X_train_scaled.csv", index=True)
X_test_scaled.to_csv("../data/processed/X_test_scaled.csv", index=True)

print("Saved to data/processed/:")
print(f"  X_train.csv          {X_train.shape}")
print(f"  X_test.csv           {X_test.shape}")
print(f"  y_train.csv          {y_train.shape}")
print(f"  y_test.csv           {y_test.shape}")
print(f"  X_train_scaled.csv   {X_train_scaled.shape}")
print(f"  X_test_scaled.csv    {X_test_scaled.shape}")
print("\nThese are the full 19-feature matrices. Notebook 4 derives the "
      "Lasso-reduced Logistic Regression features from X_train_scaled directly.")

Saved to data/processed/:
  X_train.csv          (428, 19)
  X_test.csv           (107, 19)
  y_train.csv          (428,)
  y_test.csv           (107,)
  X_train_scaled.csv   (428, 19)
  X_test_scaled.csv    (107, 19)

These are the full 19-feature matrices. Notebook 4 derives the Lasso-reduced Logistic Regression features from X_train_scaled directly.
